In [36]:
import pandas as pd

In [37]:
records = []
with open("../data/interpro2go") as f:
    for line in f:
        if line.startswith("!"):
            continue
        ipr = line.split(">")[0].split(":")[1].strip().split()[0]
        go = line.strip().split(";")[-1].strip()
        desc1 = line.strip().partition(" ")[2].split(" > ")[0]
        desc2 = line.strip().partition(">")[2].strip().partition(";")[0].strip()
        records.append({"interpro_accession": ipr, "go_term": go, "desc1": desc1, "desc2": desc2})

interpro2go_df = pd.DataFrame(records)
interpro2go_df["go_desc"] = interpro2go_df["desc2"].str.replace("GO:", "")

In [38]:
raw_interpro = pd.read_csv("../data/UP000005640_9606.tsv", sep="\t", usecols = ["protein_accession", "interpro_accession"])
raw_interpro["uniprot_id"] = raw_interpro["protein_accession"].str.split("|").str[1]

raw_interpro = raw_interpro.loc[raw_interpro.interpro_accession != "-", ["uniprot_id", "interpro_accession"]]

In [40]:
final = raw_interpro.merge(
    interpro2go_df[["interpro_accession", "go_desc"]], left_on="interpro_accession", right_on="interpro_accession", how="inner"
).drop(columns=["interpro_accession"]).drop_duplicates()

final

,uniprot_id,go_desc
0,Q969Q1,zinc ion binding
3,P16455,catalytic activity
4,P16455,DNA repair
5,P16455,methylated-DNA-[protein]-cysteine S-methyltran...
15,Q9H6E5,RNA binding
...,...,...
197168,Q8TAG9,intracellular protein transport
197169,Q8TAG9,vesicle tethering involved in exocytosis
197172,Q96RQ9,oxidoreductase activity
197176,O75808,calcium-dependent cysteine-type endopeptidase ...


In [ ]:
final.to_json("../data/uniprot_id_to_go.json", orient="records", indent=1)